# Minilink showcase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/showcase_minilink.ipynb)

**Write the equations once. Simulate, analyze, control, plan, optimize, learn.**

Minilink is an open-source Python toolbox for dynamical systems and control. A model
is three functions of the state `x`, the input `u`, the time `t` and the parameters `p`:

    dx/dt = f(x, u, t; p)      dynamics
    y     = h(x, u, t; p)      outputs, default y = x
    T     = tf(x, u, t; p)     body poses, for animation

Everything else in the library is a tool that takes such a model, or an object built
from one. This notebook walks a ladder of tools on a few catalog plants, and ends
with what the same model gives once it is compiled and differentiated.

1. [Ten lines](#1.-Ten-lines): a plant, a controller, a loop
2. [Blocks, wires, sample rates](#2.-Blocks,-wires,-sample-rates): sources, named signals, a digital controller
3. [Write your own System](#3.-Write-your-own-System): `f` and `tf`, then graphics for free
4. [A diagram is a System](#4.-A-diagram-is-a-System): Bode, margins, step response, root locus
5. [State feedback on the cart-pole](#5.-State-feedback-on-the-cart-pole): LQR at the upright point
6. [One problem, three planners](#6.-One-problem,-three-planners): value iteration, RRT and collocation on one phase plane
7. [Trajectory optimization](#7.-Trajectory-optimization): cart-pole swing-up
8. [Model predictive control](#8.-Model-predictive-control): a digital controller on a continuous plant
9. [Reinforcement learning](#9.-Reinforcement-learning): the plant as a Gymnasium environment
10. [A six-axis robot](#10.-A-six-axis-robot): UR5 in 3D
11. [Differentiable](#11.-Differentiable): exact Jacobians, batched rollouts, gains tuned through the simulation
12. [Where next](#12.-Where-next)

Each section is a few lines of code. The *under the hood* notes point to the
per-package intro notebooks (`00_core` … `10_graphical`) for the details.

**Run it** locally in the `minilink` conda env (see the repository README), or on
Colab: the next cell clones the repository.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import importlib.util
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

_OPTIMIZER_METHOD = (
    "ipopt" if importlib.util.find_spec("cyipopt") is not None else "scipy_slsqp"
)

## 1. Ten lines

A catalog plant, a controller, and `@` to close the loop. Displaying the diagram
draws its block diagram; `compute_trajectory` integrates the dynamics and keeps the
result on the object for plotting and animation.

In [ ]:
import numpy as np

from minilink import ImpedanceController, Pendulum

plant = Pendulum()
plant.x0[0] = 2.0  # initial angle [rad]
plant.params["l"] = 5.0  # rod length [m]

ctl = ImpedanceController(Kp=100.0, Kd=10.0)  # u = Kp (r - theta) - Kd dtheta
diagram = ctl @ plant  # plant.y -> ctl.y, ctl.u -> plant.u
diagram

In [ ]:
diagram.compute_trajectory(tf=10.0)
diagram.plot_trajectory()

In [ ]:
diagram.animate(renderer="plotly")

`plot_phase_plane` draws the vector field of `f` with the simulated path on top: one
picture of what the equations do.

In [ ]:
plant.plot_phase_plane()

## 2. Blocks, wires, sample rates

A source block feeds the reference and `>>` chains outputs into inputs. Every wire
keeps its name, so any internal signal can be plotted as `subsystem:port`. A digital
controller runs at its own sample time: `ctl % dt` samples the controller and holds
its output between ticks, while the plant stays continuous.

*Under the hood:* `+` adds subsystems without wiring, `>>` and `@` connect them, and
`.autowire()` connects matching port names ([00_core](00_core.ipynb)). The sampled loop
is a `HybridDiagram`: a `Computer` ticks the discrete side, the continuous plant is
integrated between ticks ([06_hybrid](06_hybrid.ipynb)).

In [ ]:
from minilink import Step

step = Step(final_value=np.array([1.0]), step_time=2.0)  # reference r(t)
plant.x0[0] = 0.0
loop = step >> ctl @ plant
loop.plot_diagram()

In [ ]:
loop.compute_trajectory(tf=8.0)
loop.plot_trajectory(signals=("ref:y", "sys:x", "ctl:u"))  # any wire, by name

In [ ]:
plant.x0[0] = 1.0
sampled = (ctl % 0.1) @ plant  # controller at 10 Hz, zero-order hold
sampled.compute_trajectory(tf=8.0)
sampled.plot_trajectory()

## 3. Write your own System

Subclass `DynamicSystem`, write `f` like the textbook with the coefficients in
`params`, and the plant simulates and plots. Add `tf` (body poses from the state) and a
skin (the shapes attached to each body) and it animates.

*Under the hood:* `params=None` means "use `self.params`"; any other dict overrides,
which is how tools sweep and differentiate parameters later
([02_dynamics](02_dynamics.ipynb)). Animation draws named frames of primitives
(boxes, rods, springs, meshes) through one renderer contract
([10_graphical](10_graphical.ipynb)).

In [ ]:
from minilink import DynamicSystem
from minilink.core.kinematics import translation
from minilink.graphical.animation.primitives import Box, ground_line


class MassSpringDamper(DynamicSystem):
    # m p'' + c p' + k p = u

    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=2)
        self.params = {"m": 1.0, "k": 4.0, "c": 0.3}
        self.skin = lambda sys: {
            "world": [ground_line(length=8.0)],
            "body": [Box(length_x=0.6, length_y=0.6, length_z=0.1)],
        }
        self.camera_scale = 4.0

    def f(self, x, u, t=0, params=None):
        p = self.params if params is None else params
        pos, vel = x
        acc = (u[0] - p["c"] * vel - p["k"] * pos) / p["m"]
        return np.array([vel, acc])

    def tf(self, x, u, t=0, params=None):
        return {"body": translation(x[0], 0.0, 0.0)}


msd = MassSpringDamper()
msd.x0[0] = 1.0
msd.params["c"] = 1.0  # retune without touching f
msd_loop = Step(final_value=np.array([10.0]), step_time=2.0) >> msd
msd_loop.compute_trajectory(tf=20.0)
msd_loop.plot_trajectory()

The same `tf` drives every renderer; pick the one that fits the environment.

| Renderer | Typical use |
| --- | --- |
| `plotly` | interactive HTML in notebooks |
| `meshcat` | 3D in the browser |
| `matplotlib` | inline, Agg, saved figures and GIFs |
| `pygame` | native window, and the keyboard game mode |

In [ ]:
msd_loop.animate(renderer="plotly")
# msd_loop.animate(renderer="meshcat")
# msd_loop.animate(renderer="matplotlib")
# msd_loop.animate(renderer="pygame")

In [ ]:
PLAY = False  # set True locally: opens a window, the keyboard drives u in real time
if PLAY:
    msd.game()

## 4. A diagram is a System

A closed loop is a `System`, so the analysis tools apply to it as to a plant.
Linearize about an operating point and read the frequency response; the classical
loop tools take the same objects: `C >> G` is the loop gain, `C @ G` closes the loop
through the error junction `e = r - y`.

*Under the hood:* every analysis verb reads `tool(sys, x_bar, u_bar, ...)` and is
exact under JAX when the plant traces, finite differences otherwise
([04_analysis](04_analysis.ipynb), [frequency-domain tools](../teaching/frequency_domain_tools.ipynb)).

In [ ]:
from minilink import PID

G = Pendulum()
G.params["d"] = 1.0  # a little damping
G.x0 = np.array([0.0, 0.0])  # operating point: hanging down

G.plot_bode()  # channel theta / tau of the plant linearized at x0

In [ ]:
C = PID(Kp=20.0, Ki=10.0, Kd=2.0, tau=0.05)
L = C >> G  # loop gain L = C G
print("margins of L:", L.margins())
L.plot_bode()

In [ ]:
T = C @ G  # closed loop r -> theta
T.plot_step_response()

In [ ]:
from minilink import InvertedPendulum, Lead

upright = InvertedPendulum()
upright.x0 = np.array([0.0, 0.0])
(Lead(K=1.0, z=2.0, p=20.0) >> upright).plot_root_locus()

## 5. State feedback on the cart-pole

`lqr_at_operating_point` linearizes the plant at an equilibrium and returns a
`StateFeedbackController` ready for `@`. The cart-pole is the plant with the nicest
animation, so the next sections keep coming back to it.

*Under the hood:* the A and B matrices come from `jacobian("f", "x")` and
`jacobian("f", "u")`; the gain solves the continuous Riccati equation
([03_control](03_control.ipynb)).

In [ ]:
from minilink import CartPole, lqr_at_operating_point

cartpole = CartPole()
x_up = np.array([0.0, np.pi, 0.0, 0.0])
lqr_ctl = lqr_at_operating_point(cartpole, x_up, Q=np.eye(4), R=np.array([[1.0]]))
lqr_loop = lqr_ctl @ cartpole

cartpole.x0 = np.array([-3.0, np.pi - 0.3, 0.0, 0.0])
lqr_loop.compute_trajectory(tf=8.0)
lqr_loop.plot_trajectory()

In [ ]:
lqr_loop.animate(renderer="plotly")

## 6. One problem, three planners

A `PlanningProblem` is a system, a cost and boundary sets. The three planners below
take the same object: value iteration on a state grid, a kinodynamic tree search,
and direct collocation. Their answers are drawn on the same phase plane at the end.

*Under the hood:* value iteration returns a `LookupTableController`, which is a
`System` and closes the loop with `@`. RRT grows a tree with bang-bang torques
taken from the input bounds. Collocation transcribes the problem into a `MathematicalProgram` solved by
an `Optimizer` ([09_planning](09_planning.ipynb), [08_optimization](08_optimization.ipynb)).

In [ ]:
from minilink import (
    BallSet,
    DynamicProgrammingPlanner,
    PlanningProblem,
    QuadraticCost,
    RRTPlanner,
    TrajectoryOptimizationPlanner,
)
pend = Pendulum()
pend.inputs["u"].lower_bound[:] = -5.0
pend.inputs["u"].upper_bound[:] = 5.0
pend.state.lower_bound = np.array([-2.0 * np.pi, -12.0])
pend.state.upper_bound = np.array([2.0 * np.pi, 12.0])

x_down, x_up = np.array([0.0, 0.0]), np.array([np.pi, 0.0])
problem = PlanningProblem(
    sys=pend,
    x_start=x_down,
    x_goal=x_up,
    Xf=BallSet(x_up, 0.2),
    tf=4.0,
    cost=QuadraticCost.from_system(pend, Q=np.eye(2), R=np.eye(1), xbar=x_up),
)

In [ ]:
vi = DynamicProgrammingPlanner(problem, x_grid=(101, 101), u_grid=(11,), dt=0.05, out_of_bound_cost=500.0)
vi.solve()
vi.plot_cost2go(jmax=500.0)

In [ ]:
vi_loop = vi.get_controller() @ pend  # the policy is a controller
vi_traj = vi_loop.compute_trajectory(tf=10.0)  # pumping from rest takes a few swings
vi.plot_policy(trajectory=vi_traj)

In [ ]:
rrt = RRTPlanner(problem, seed=0, max_nodes=5000)  # bang-bang inputs from the bounds
rrt_traj = rrt.solve().trajectory
rrt.plot_tree(x_axis=0, y_axis=1)

In [ ]:
opt = TrajectoryOptimizationPlanner(
    problem, n_steps=40, transcription="direct_collocation", optimizer_method=_OPTIMIZER_METHOD
)
opt_traj = opt.solve().trajectory
opt.plot_solution(signals=("x", "u"))

In [ ]:
import matplotlib.pyplot as plt

phase = pend.plot_phase_plane(show=False)
for traj, label, style in (
    (opt_traj, "collocation", dict(color="C2", lw=3)),
    (rrt_traj, "RRT", dict(color="C1", lw=2)),
    (vi_traj, "value iteration", dict(color="C0", lw=1.5, ls="--")),
):
    phase.axes.plot(traj.x[0], traj.x[1], label=label, **style)
phase.axes.legend()
plt.show()

## 7. Trajectory optimization

Cart-pole swing-up by direct collocation. `compile_backend="jax"` gives the
transcription exact gradients of the dynamics defects, and the solution animates
through the plant like any trajectory.

*Under the hood:* `TrajectoryOptimizationPlanner` builds the `MathematicalProgram`
(decision vector, cost, defect constraints) and hands it to `Optimizer`, SciPy or
Ipopt. `transcription="shooting"` and `"multiple_shooting"` use the same problem
([09_planning](09_planning.ipynb)).

In [ ]:
jax_cartpole = CartPole()
jax_cartpole.inputs["u"].lower_bound[0] = -10.0
jax_cartpole.inputs["u"].upper_bound[0] = 10.0
x_up_cp = np.array([0.0, np.pi, 0.0, 0.0])

swing_up = PlanningProblem(
    sys=jax_cartpole,
    x_start=np.array([-2.0, 0.0, 0.0, 0.0]),
    x_goal=x_up_cp,
    tf=4.0,
    cost=QuadraticCost.from_system(jax_cartpole, Q=np.diag([1.0, 1.0, 0.0, 0.0]), xbar=x_up_cp),
)

planner = TrajectoryOptimizationPlanner(
    swing_up,
    n_steps=20,
    transcription="direct_collocation",
    compile_backend="jax",
    optimizer_method=_OPTIMIZER_METHOD,
    verbose=True,
)
swing_traj = planner.solve().trajectory
planner.plot_solution(signals=("x", "u"))

In [ ]:
jax_cartpole.animate(swing_traj, renderer="plotly")

## 8. Model predictive control

A digital controller on a continuous plant. The `ModelPredictiveController` wraps a
trajectory-optimization planner and re-solves it at each tick from the measured
state; `mpc @ plant` builds the sampled loop with zero-order hold, and the continuous
plant is integrated between ticks.

*Under the hood:* the planner's problem is re-parametrized online (`x_start`, the
reference) without rebuilding the program; `warm_start=True` reuses the previous
solution ([06_hybrid](06_hybrid.ipynb), `examples/demos/mpc/`).

In [ ]:
from minilink import BicycleDynRate
from minilink.control.mpc import ModelPredictiveController, mpc_animation_overlays

U_TARGET = 4.0
car = BicycleDynRate()
r_r = car.params["r_r"]
x_ref = np.array([0.0, 0.0, 0.0, U_TARGET, 0.0, 0.0, U_TARGET / r_r, 0.0])
x0 = np.array([0.0, 3.0, 0.0, 0.8 * U_TARGET, 0.0, 0.0, 0.8 * U_TARGET / r_r, 0.0])
car.x0 = x0.copy()

mpc_planner = TrajectoryOptimizationPlanner(
    PlanningProblem(
        sys=car,
        tf=2.0,
        x_start=x0,
        cost=QuadraticCost.from_system(
            car,
            Q=np.diag([0.0, 12.0, 18.0, 0.5, 4.0, 6.0, 0.1, 100.0]),
            R=np.diag([1.0, 25.0]),
            S=np.diag([0.0, 30.0, 40.0, 2.0, 12.0, 18.0, 0.1, 100.0]),
            xbar=x_ref,
        ),
    ),
    n_steps=5,
    transcription="direct_collocation",
    compile_backend="jax",
    optimizer_method="scipy_slsqp",
    optimizer_options={"maxiter": 10, "ftol": 1.0},
)

mpc = ModelPredictiveController(mpc_planner, dt_mpc=0.2, warm_start=True, verbose=False)
mpc_loop = mpc @ car
result = mpc_loop.compute_trajectory(tf=10.0, x0_plant=x0, plant_dt_inner=0.02, compile_backend="jax")
mpc_loop.plot_trajectory()

In [ ]:
mpc_loop.animate(overlays=mpc_animation_overlays(result, mpc_planner, reference_pad=20.0))

## 9. Reinforcement learning

The same plant and cost become a Gymnasium environment: `reward = -g(x, u) dt`, the
step is one compiled RK4 call. Train with any agent; the trained policy comes back as
an `SB3Controller`, a `System` that closes the loop with `@`.

*Under the hood:* the [VI vs LQR vs PPO](../teaching/pendulum_swing_up_vi_vs_lqr_vs_ppo.ipynb)
and [drone](../teaching/drone_ppo_learn_to_fly.ipynb) notebooks train PPO on these
environments (`pip install minilink[rl]`, or the conda env).

In [ ]:
try:
    from minilink.interfaces.gymnasium import Sys2Gym

    env = Sys2Gym(pend, problem.cost, dt=0.05, tf=5.0)
    obs, info = env.reset(seed=0)
    obs, reward, terminated, truncated, info = env.step(np.array([1.0]))
    print(env.observation_space, env.action_space, f"reward = {float(reward):.3f}")
except ImportError:
    print("gymnasium is not installed: pip install gymnasium")

## 10. A six-axis robot

A UR5 with articulated-body dynamics is the same kind of object as the pendulum.
`closed_loop_qdq` wires a joint-space controller to the mechanical system's `(q, dq)`
outputs; the joint impedance controller compensates gravity. meshcat opens a 3D
viewer (a URL locally, inline on Colab); `renderer="plotly"` with `is_3d=True` is the
fallback.

*Under the hood:* `MechanicalSystem` plants implement `H(q)`, `C(q, dq)`, `g(q)` and
the forward kinematics; the UR5 uses recursive Newton-Euler and articulated-body
algorithms, and compiles on both backends ([02_dynamics](02_dynamics.ipynb),
[robot equations of motion](../teaching/articulated_robot_eom.ipynb)).

In [ ]:
from minilink import JointImpedance, UR5Manipulator, closed_loop_qdq

arm = UR5Manipulator()
q0 = np.array([0.0, -np.pi / 2, 0.0, -np.pi / 2, 0.0, 0.0])
q1 = np.array([1.35, -1.2, 1.55, -1.4, 0.35, 0.25])
arm.x0 = arm.q2x(q0, np.zeros(6))

arm_ctl = JointImpedance(arm, gravity_comp=True)
arm_ctl.params["Kp"] = np.array([10.0, 60.0, 20.0, 0.5, 0.1, 0.003])
arm_ctl.params["Kd"] = np.array([3.0, 25.0, 8.0, 0.2, 0.05, 0.003])

arm_loop = Step(initial_value=q0, final_value=q1, step_time=1.0) >> closed_loop_qdq(arm_ctl, arm)
arm_loop.compute_trajectory(tf=6.0, n_steps=180, compile_backend="jax")
arm_loop.plot_trajectory()

In [ ]:
arm_loop.animate(renderer="meshcat", is_3d=True)

## 11. Differentiable

The same `f` traces under JAX. `jacobian` reads exact derivatives of the dynamics with
respect to the state, the input, or each physical parameter. `compile(backend="jax")`
returns one flat evaluator whose primitives run a family of rollouts in one call, and
whose trace tier can be differentiated through a whole closed-loop simulation.

*Under the hood:* the evaluator exposes `f`, `rk4_step`, `rk4_integrate_zoh`,
`rollout_batch` in a jit tier (what simulators and planners call) and a trace tier
(`f_trace`, `rk4_step_p`, … what you differentiate inside your own `jit`);
JAX evaluators use float64 ([07_compile](07_compile.ipynb),
[JAX showcase](showcase_jax.ipynb)).

In [ ]:
pdiff = Pendulum()
pdiff.params["I"] = 0.0  # point mass on a massless rod
pdiff.params["d"] = 0.2  # viscous damping
x_bar, u_bar = np.array([0.5, 1.0]), np.array([0.0])

A = pdiff.jacobian("f", "x", x_bar, u_bar)  # exact under JAX
dfdp = pdiff.jacobian("f", "params", x_bar, u_bar)  # one sensitivity vector per parameter
print("A = df/dx =\n", A.round(3))
for key, column in dfdp.items():
    print(f"d(dx/dt)/d{key:<8} = {np.asarray(column).round(3)}")

In [ ]:
import time

ev = pdiff.compile(backend="jax")
N, STEPS, DT = 1000, 1000, 0.005
x0s = np.column_stack([np.linspace(-3.0, 3.0, N), np.zeros(N)])  # 1000 initial angles

xs = np.asarray(ev.rollout_batch(x0s, n_steps=STEPS, dt=DT))  # first call compiles
t0 = time.perf_counter()
xs = np.asarray(ev.rollout_batch(x0s, n_steps=STEPS, dt=DT))
t_batch = time.perf_counter() - t0

ev_np = pdiff.compile(backend="numpy")
t0 = time.perf_counter()
x = x0s[0]
for k in range(STEPS):
    x = ev_np.rk4_step(x, np.zeros(1), k * DT, DT)
t_one = time.perf_counter() - t0

print(f"{N} rollouts x {STEPS} RK4 steps, compiled batch : {1e3 * t_batch:7.1f} ms")
print(f"1 rollout step by step in Python, x{N}          : {1e3 * t_one * N:7.0f} ms")

# a family of rod lengths in one call: one value per rollout in params
lengths = np.linspace(0.5, 2.0, 5)
family = dict(pdiff.params, l=lengths, d=0.0)
xs_l = np.asarray(ev.rollout_batch(np.tile([1.0, 0.0], (5, 1)), n_steps=STEPS, dt=DT, params=family))
t = DT * np.arange(STEPS + 1)
for x_l, ell in zip(xs_l, lengths):
    plt.plot(t, x_l[:, 0], label=f"l = {ell:.2f} m")
plt.xlabel("t [s]")
plt.ylabel("theta [rad]")
plt.legend()
plt.show()

Gradients go through the whole closed loop. Below, the three gains of a PID-type
controller are tuned by gradient descent on the tracking error of a 10 s simulation:
the loss is a rollout of the compiled diagram, and `jax.grad` returns its derivative
with respect to the gains.

In [ ]:
import jax
import jax.numpy as jnp

from minilink import ImpedanceIntegralController

pid_plant = Pendulum()
pid = ImpedanceIntegralController()
pid.params = {"Kp": 1.0, "Ki": 0.0, "Kd": 1.0}
pid_loop = pid @ pid_plant
pid_loop.inputs["r"].nominal_value = np.array([1.0])  # setpoint theta = 1 rad

ev_loop = pid_loop.compile(backend="jax")
theta_idx = pid_loop.state_index["sys"][0]
x0, r, dt = jnp.array(pid_loop.x0), jnp.array([1.0]), 0.05
ts = jnp.arange(200) * dt


def tracking_loss(gains):
    params = {"ctl": {"Kp": gains[0], "Ki": gains[1], "Kd": gains[2]}, "sys": pid_plant.params}

    def step(x, t):
        x_next = ev_loop.rk4_step_p(x, r, t, dt, params)
        return x_next, (r[0] - x_next[theta_idx]) ** 2

    _, errors = jax.lax.scan(step, x0, ts)
    return jnp.mean(errors)


loss_and_grad = jax.jit(jax.value_and_grad(tracking_loss))
gains = jnp.array([1.0, 0.0, 1.0])
for i in range(301):
    loss, grad = loss_and_grad(gains)
    if i % 100 == 0:
        print(f"iter {i:3d}  loss {float(loss):.4f}  Kp, Ki, Kd = {np.round(np.asarray(gains), 2)}")
    gains = gains - 1.0 * grad

pid.params = {k: float(v) for k, v in zip(("Kp", "Ki", "Kd"), gains)}
pid_loop.compute_trajectory(tf=10.0)
pid_loop.plot_trajectory()

## 12. Where next

- [JAX showcase](showcase_jax.ipynb): write `f` once, get every gradient
- [Intro series](./): `00_core` … `10_graphical`, one notebook per package
- [Teaching notebooks](../teaching/): swing-up by value iteration, LQR and PPO;
  frequency-domain tools; robot equations of motion; rollout gradients
- [Examples index](../../README.md): demos and projects by chapter
- [DESIGN.md](../../../DESIGN.md) and [ROADMAP.md](../../../ROADMAP.md): contracts and plan of record